#### The **Silver layer** transforms raw data from the Bronze layer into clean, structured, and reliable datasets. 

- This stage focuses on data quality and consistency by applying validation rules, removing duplicates, handling missing values, and standardizing formats.

- Business logic is applied to filter relevant records and enrich data through joins or derived fields. 

- The processed data is then stored in the Silver zone, making it suitable for downstream analytics and further aggregation in the Gold layer.

In [0]:
from pyspark.sql.functions import *

### Establish Connection

In [0]:
spark.conf.set("fs.azure.account.key.sa1storage.dfs.core.windows.net", "G1RfRXHyK97KaTGSJXAD1RqeXkcX0WRYxtUi4g/pUZx7GxFi3MWAEvU7uU9yh7I27Syh/IUORuea+AStsT8Lzw==")

In [0]:
filename = dbutils.fs.ls("abfss://bronze@sa1storage.dfs.core.windows.net/rawdata")[0].name

In [0]:
df = spark.read.parquet(f"abfss://bronze@sa1storage.dfs.core.windows.net/rawdata/{filename}")
df.limit(20).display()

OrderID,OrderDate,CustomerID,CustomerName,CustomerEmail,Country,ProductID,ProductCategory,ProductName,Quantity,UnitPrice,TotalPrice,SalesRegion
1,2024-12-09,1849,Johnny Thompson,christinebailey@robinson.biz,British Virgin Islands,599,Electronics,Table,2,411.44,822.88,West
2,2024-04-18,1463,Christina Bowman,jennifersalazar@yahoo.com,Korea,524,Sports,Laptop,10,820.20,8202.00,North
3,2024-02-10,1308,Sean Harris,harold74@carrillo.com,American Samoa,583,Clothing,Table,5,922.09,4610.45,North
4,2025-04-07,1366,Tina Munoz,pcarlson@morrison.com,Togo,596,Sports,T-Shirt,4,814.39,3257.56,East
5,2024-05-31,1314,James Ford,qromero@castillo-jackson.com,Tonga,549,Electronics,Perfume,10,547.25,5472.50,South
6,2025-06-12,1525,Elizabeth Cunningham,karen47@lee-cox.com,Azerbaijan,537,Clothing,Shoes,5,978.02,4890.10,West
7,2023-07-04,1517,Lisa Ponce,mark14@burns.biz,Slovenia,560,Electronics,Table,1,330.20,330.20,West
8,2024-12-10,1577,Suzanne Bowen,bsmith@hotmail.com,Netherlands Antilles,565,Clothing,Bicycle,6,570.58,3423.48,South
9,2023-11-19,1825,Andrew Johnson,carolyn00@hotmail.com,Uruguay,514,Beauty,Tennis Racket,6,374.67,2248.02,West
10,2024-03-19,1514,Brenda Meyers,laurendaniels@yahoo.com,Saudi Arabia,593,Furniture,Perfume,7,176.95,1238.65,East


### Data Exploration

In [0]:
df.printSchema()

root
 |-- OrderID: integer (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- CustomerEmail: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- ProductID: integer (nullable = true)
 |-- ProductCategory: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: decimal(18,2) (nullable = true)
 |-- TotalPrice: decimal(18,2) (nullable = true)
 |-- SalesRegion: string (nullable = true)



#### - Converting to titlecase - initcap

In [0]:
# df = df.withcolumn("Country", initcap(df['Country']))

#### - Removing rows where full row is empty

In [0]:
df = df.dropna(how = 'all')
df.limit(20).display()

OrderID,OrderDate,CustomerID,CustomerName,CustomerEmail,Country,ProductID,ProductCategory,ProductName,Quantity,UnitPrice,TotalPrice,SalesRegion
1,2024-12-09,1849,Johnny Thompson,christinebailey@robinson.biz,British Virgin Islands,599,Electronics,Table,2,411.44,822.88,West
2,2024-04-18,1463,Christina Bowman,jennifersalazar@yahoo.com,Korea,524,Sports,Laptop,10,820.20,8202.00,North
3,2024-02-10,1308,Sean Harris,harold74@carrillo.com,American Samoa,583,Clothing,Table,5,922.09,4610.45,North
4,2025-04-07,1366,Tina Munoz,pcarlson@morrison.com,Togo,596,Sports,T-Shirt,4,814.39,3257.56,East
5,2024-05-31,1314,James Ford,qromero@castillo-jackson.com,Tonga,549,Electronics,Perfume,10,547.25,5472.50,South
6,2025-06-12,1525,Elizabeth Cunningham,karen47@lee-cox.com,Azerbaijan,537,Clothing,Shoes,5,978.02,4890.10,West
7,2023-07-04,1517,Lisa Ponce,mark14@burns.biz,Slovenia,560,Electronics,Table,1,330.20,330.20,West
8,2024-12-10,1577,Suzanne Bowen,bsmith@hotmail.com,Netherlands Antilles,565,Clothing,Bicycle,6,570.58,3423.48,South
9,2023-11-19,1825,Andrew Johnson,carolyn00@hotmail.com,Uruguay,514,Beauty,Tennis Racket,6,374.67,2248.02,West
10,2024-03-19,1514,Brenda Meyers,laurendaniels@yahoo.com,Saudi Arabia,593,Furniture,Perfume,7,176.95,1238.65,East


#### - Checking duplicate data

In [0]:
df = df.drop_duplicates()
df.limit(20).display()

OrderID,OrderDate,CustomerID,CustomerName,CustomerEmail,Country,ProductID,ProductCategory,ProductName,Quantity,UnitPrice,TotalPrice,SalesRegion
38,2024-04-10,1696,George Wright,ibarnes@davis.com,Guatemala,579,Sports,Phone,5,583.72,2918.60,Central
433,2025-02-06,1121,Rachael Paul,rachael01@richmond.com,Bahrain,536,Furniture,Laptop,8,178.08,1424.64,Central
464,2024-09-24,1939,Matthew Lane,perryjoseph@gmail.com,Czech Republic,538,Clothing,Laptop,3,891.24,2673.72,West
618,2025-02-09,1354,Douglas Park,erowland@davis-doyle.info,Bulgaria,573,Furniture,Shoes,5,182.88,914.40,Central
632,2023-10-28,1187,Kelly Fisher,iparker@yahoo.com,Panama,539,Furniture,Shoes,9,782.59,7043.31,West
885,2023-10-09,1990,Gary Ray,christopherwilson@gmail.com,Saint Martin,552,Clothing,Laptop,1,842.54,842.54,East
969,2023-08-22,1249,Christine Curtis,thomas75@yahoo.com,Northern Mariana Islands,598,Electronics,Perfume,4,145.78,583.12,South
6,2025-06-12,1525,Elizabeth Cunningham,karen47@lee-cox.com,Azerbaijan,537,Clothing,Shoes,5,978.02,4890.10,West
112,2023-10-02,1442,Michelle Craig,christianguerrero@hotmail.com,Nepal,553,Beauty,Table,9,84.37,759.33,North
311,2024-11-21,1196,Rhonda Hardy,alexis46@hotmail.com,Mongolia,545,Electronics,Bicycle,4,13.15,52.60,West


#### - Moving to 'SILVER' Layer

In [0]:
df.write.format('parquet').mode('overwrite').option("path", "abfss://silver@sa1storage.dfs.core.windows.net/cleanseddata").save()